Name: Oliver Vesey
<br>
RAN: 101043434
<br>
$\beta_1 = 27$, $\beta_2 = -13$, $\beta_3 = 15$
<br>
$\sigma_1 = 6$, $\sigma_2 = 12$, $\sigma_3 = 18$

In [3]:
import numpy as np
import pandas as pd

class MLR:

    def __init__(self, x, y):
        
        self.X = np.array(x, dtype=float) #Saves the x values entered as a NumPy matrix
        self.y = np.array(y, dtype=float).reshape(-1, 1) #Saves the y values and turns it into a column vector with n rows (amount of y values).
        self.n, self.k = self.X.shape #Saving self.n as the number of observations and self.k as the number of predictors using '.shape'.
        self.y_bar = float(np.mean(self.y))

#Method 1: 

    def LS_est(self):
        
        self.X_aug = np.column_stack((np.ones(self.n), self.X)) #Builds the X matrix by adding ones in the left-most column.
        XtX = self.X_aug.T @ self.X_aug #Computing (X'X)^(-1) X'y (beta_hat) step by step.
        XtX_inv = np.linalg.inv(XtX)
        Xty = self.X_aug.T @ self.y
        self.beta_hat = XtX_inv @ Xty

        return self.beta_hat #Returns a column vector containing alpha first, followed by the the beta(s) (slope coefficents).

#Method 2: 

    def R2(self):
      
        self.LS_est() #Run LS_est to find alpha and beta values in self.beta_hat.

        y_hat = self.X_aug @ self.beta_hat #Find the fitted values by computing (X_aug)(beta_hat).
        
        SSE = 0.0
        SST = 0.0
        for i in range(self.n): #Summing to find SSE and SST respectively for the R^2 value
            SSE += (self.y[i,0] - y_hat[i,0])**2 #Using index [i,0] for the ith entry of column 0 (first and only column of self.y and y_hat with 'self.n many' rows).
            SST += (self.y[i,0] - self.y_bar)**2  
        return (1 - SSE/SST)

# Method 3 

    def MSE(self):
        
        self.LS_est() #Again run LS.est for the alpha and beta values.
        
        y_hat = self.X_aug @ self.beta_hat #Calculate the fitted values again.

        SSE = 0.0
        for i in range(self.n): #Summing to find SSE value again.
            SSE += (self.y[i,0] - y_hat[i,0])**2 
          
        MSE = SSE/(self.n - self.k - 1) #Running this will return the MSE value as a float.

        return MSE

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

#Generating Synthetic Data:

RAN=101043434
np.random.seed(RAN) 
n=40

alpha=2025

beta1 = 27   #Beta values as they were in SP2 with the addition of beta3 = 15.
beta2 = -13
beta3 = 15

x1=np.random.uniform(1,2,n)  #x1 and x2 as they were in SP2, with the addition of x3 (similar to x1 and x2).
x2=np.random.uniform(3,6,n) 
x3=np.random.uniform(5,7,n) 
x = np.c_[x1,x2,x3] #Making x the matrix with these x_i's as its columns.


sigmas = [6,12,18] #Three different sigma values to test the code on.


for i in sigmas:
    error = i*np.random.normal(0,1,n)  #Creating the different error and corresponding y values for each sigma value.
    
    y = alpha+(beta1*x1)+(beta2*x2)+(beta3*x3)+error 
    Y=y.reshape(-1,1)  #Converting said y values and reshaping to column vector with shape (40x1).
    
    model = MLR(x,Y)  #First finding the different values with my code.
    print("Simulation of the data with sigma =",i,", with my code:")
    print("")
    coeffs = model.LS_est()
    print(coeffs[1,0], ", ", coeffs[2,0]," ,",  coeffs[3,0]," ,",coeffs[0,0]) #Printing in the order beta1, beta2, beta3, alpha.
    print(model.MSE(),", ", (model.MSE()**0.5)) #Printing the MSE value, followed by te RMSE (estimate of sigma).
    print(model.R2()) #Printing R^2 from my code.
    print("")
    
    print("Simulation of the data with sigma =",i,", with sklearn:")
    print("")
    model2 = LinearRegression()  #Calling an instance of the Linear Regression class and fitting it to our x and Y.
    model2.fit(x,Y)
    
    Beta = model2.coef_       #Finding our alpha and beta values using the respective methods from the LinearRegression class.
    Alpha = model2.intercept_

    print(Beta[0,0], ", ", Beta[0,1]," ,",  Beta[0,2]," ,",Alpha[0]) #Printing the estimates in the same order as my code for clarity.

    y_hat=model2.predict(x) #Finding the fitted values for x.

    mse1=mean_squared_error(y,y_hat)
    MSE1=mse1*len(y)/(len(y)-4)
    print(MSE1,", ", MSE1**0.5)       #Printing the MSE value, followed by the RMSE (estimate of sigma).
    
    print(model2.score(x,y))  #Print R^2 value directly using the method of the LinearRegression class.
    print("")
    print("")



Simulation of the data with sigma = 6 , with my code:

24.18034692754736 ,  -14.098939298218056  , 15.600144020041625  , 2030.7931599652802
31.073937190451332 ,  5.574400164183706
0.9101188436657334

Simulation of the data with sigma = 6 , with sklearn:

24.180346927522347 ,  -14.098939298216978  , 15.600144020036497  , 2030.7931599652911
31.073937190450906 ,  5.574400164183668
0.9101188436657346


Simulation of the data with sigma = 12 , with my code:

21.352260197549185 ,  -13.716900935737385  , 13.820564418689173  , 2044.3396848118573
130.40227592322984 ,  11.41938159110334
0.6751681859910054

Simulation of the data with sigma = 12 , with sklearn:

21.35226019751457 ,  -13.716900935733447  , 13.82056441868571  , 2044.3396848119314
130.4022759232298 ,  11.419381591103338
0.6751681859910055


Simulation of the data with sigma = 18 , with my code:

36.551454135318636 ,  -10.949712288455657  , 9.691364277179673  , 2035.7168681610783
317.4853149873989 ,  17.818117605050173
0.418674915766

## Results and Comparison of My Code vs sklearn:
<br>
Running the two cells of code above will generate synthetic data for three predictors. The code makes estimates for $\beta_1$, $\beta_2$, $\beta_3$, and $\alpha$, followed by the $MSE$ and the $RMSE$ (this is the estimate for $\sigma$), and lastly, the $R^2$ value. In this order, the report shows what my code produces, followed by what sklearn produces for $\sigma = 6$, $\sigma = 12$, and $\sigma = 18$ respectively.
<br>

<br>
For all three of my different $\sigma$ values, my least-squares estimates ($\beta_1$, $\beta_2$, $\beta_3$, and $\alpha$), $\sigma$ estimation, and $R^2$ value calculated by my code and the in built in sklearn, are all extremely close to one another, being identical to roughly 9dp. 
<br>

<br>
We can observe that the $\sigma$ estimates are all very close to the $\sigma$ value given, within a range of $< 0.6$. However, as $\sigma$ increases in size our least-squares estimates get weaker and our $R^2$ value gets smaller. For example, with our smallest value ($\sigma = 6$), we get:

$$ \beta_1 \approx 24.2, \beta_2 \approx -14.1, \beta_3 \approx 15.6,  \alpha \approx 2030.8 , R^2 \approx 0.91$$
This is quite close to our given values of $\beta_1 = 27$, $\beta_2 = -13$, $\beta_3 = 15$, $\alpha = 2025$. But with $\sigma$ at its largest value ($\sigma = 18$), we get:

$$ \beta_1 \approx 36.6, \beta_2 \approx -10.9, \beta_3 \approx 9.7,  \alpha \approx 2035.7, R^2 \approx 0.41$$
<br>

These estimates aren't horrible, but they are clearly weaker estimates than before.
